# Phase 2: Medical Domain Fine-Tuning (QLoRA)

**Dataset:** `medalpaca/medical_meadow_medqa` — ~10K USMLE-style 4-option clinical MCQs  
**Base model:** `mistralai/Mistral-7B-Instruct-v0.2`  
**Method:** QLoRA (4-bit quantization + LoRA adapters)  
**GPU required:** T4 (16GB) — available free on Kaggle  
**Estimated runtime:** ~90 minutes (2 epochs, 9K training samples)

## What this notebook does
1. Installs dependencies
2. Loads and explores the MedQA dataset
3. Formats samples into Mistral chat template
4. Trains a QLoRA adapter (2 epochs)
5. Evaluates MCQ accuracy on held-out test split
6. Pushes the adapter to HuggingFace Hub

## Why medical domain fine-tuning?
Base Mistral-7B-Instruct is a generalist. It knows medical facts but:
- Doesn't reliably pick a single answer letter (A/B/C/D) in MCQ format
- Misses domain-specific clinical reasoning patterns
- Fine-tuning on MedQA teaches the MCQ answer format + improves clinical accuracy

## 0. Kaggle Setup

Before running:
1. Enable GPU: **Settings → Accelerator → GPU T4 x2** (or T4 x1)
2. Enable Internet: **Settings → Internet → On** (needed to download model + dataset)
3. Add HuggingFace token as a secret: **Add-ons → Secrets → HF_TOKEN** (for pushing to Hub)

In [ ]:
# Check GPU availability
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

## 1. Install Dependencies

In [ ]:
# Standard QLoRA stack — pinned to known-compatible versions.
# Avoids Unsloth's monkey-patching issues. Works on Kaggle CUDA 13
# because bitsandbytes 0.45+ uses dynamic CUDA library loading.
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "bitsandbytes==0.45.3",
     "transformers==4.46.3",
     "peft==0.13.2",
     "trl==0.12.2",
     "accelerate==1.1.1",
     "datasets==3.0.0",
     "huggingface_hub",
     "pyyaml",
     "scikit-learn",
    ],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "")
print(result.stderr[-500:] if result.stderr else "")
print("\nDone. Now: Kernel → Restart → Run All.")

In [ ]:
import transformers, peft, trl, bitsandbytes, accelerate, datasets
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"trl:          {trl.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"datasets:     {datasets.__version__}")

In [ ]:
import torch

# ─── Model ───────────────────────────────────────────────────
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

# ─── Dataset ─────────────────────────────────────────────────
DATASET_NAME = "medalpaca/medical_meadow_medqa"
TEST_SIZE    = 0.10   # 10% held out for evaluation
MAX_SAMPLES  = None   # None = use all (~9K train after split); set e.g. 2000 for a quick test run

# ─── LoRA ────────────────────────────────────────────────────
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ─── Training ────────────────────────────────────────────────
OUTPUT_DIR          = "/kaggle/working/phase2-medical-medqa"
NUM_EPOCHS          = 2
BATCH_SIZE          = 4
GRAD_ACCUM_STEPS    = 4       # effective batch = 4 * 4 = 16
LEARNING_RATE       = 2e-4
MAX_SEQ_LENGTH      = 1024    # increased from 512 — clinical descriptions are longer
LOGGING_STEPS       = 10
SAVE_STEPS          = 200

# ─── Evaluation ──────────────────────────────────────────────
MAX_EVAL_SAMPLES    = 500     # evaluate on 500 of the held-out test samples

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 3. Load and Explore the Dataset

Understanding the data before formatting it avoids silent bugs during training.

In [ ]:
from datasets import load_dataset

print(f"Loading {DATASET_NAME}...")
raw_dataset = load_dataset(DATASET_NAME, split="train")

print(f"Total samples: {len(raw_dataset)}")
print(f"Columns: {raw_dataset.column_names}")
print()

# Inspect a few samples to understand the format
for i in [0, 1, 2]:
    sample = raw_dataset[i]
    print(f"─── Sample {i} ───")
    print(f"INPUT:\n{sample['input']}")
    print(f"\nOUTPUT:\n{sample['output']}")
    print()

In [ ]:
import re

# ─── Check for 5-option samples (A-E) ─────────────────────────
# A small number of samples have 5 options. We filter these out
# because our evaluation regex only handles A-D.
def has_5_options(sample):
    return bool(re.search(r'\bE\)', sample['input']))

five_option_count = sum(1 for s in raw_dataset if has_5_options(s))
print(f"Samples with 5 options (E): {five_option_count}")
print(f"Will filter these out before training.")

# ─── Token length distribution ────────────────────────────────
# Check what fraction of samples fit within MAX_SEQ_LENGTH=1024
lengths = [len(s['input']) + len(s['output']) for s in raw_dataset]
import numpy as np
print(f"\nChar length stats (input + output):")
print(f"  median: {int(np.median(lengths))}")
print(f"  p90:    {int(np.percentile(lengths, 90))}")
print(f"  p99:    {int(np.percentile(lengths, 99))}")
print(f"  max:    {max(lengths)}")
print(f"\nNote: 1 char ≈ 0.3-0.4 tokens for English medical text")
print(f"p99 chars / 3 ≈ {int(np.percentile(lengths, 99)) // 3} tokens → MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} is {'sufficient' if int(np.percentile(lengths, 99)) // 3 < MAX_SEQ_LENGTH else 'tight'}")

## 4. Load Tokenizer and Format Dataset

The MedQA dataset uses `input`/`output` columns (unlike Dolly's `instruction`/`response`).  
We wrap them in Mistral's `[INST]...[/INST]` chat template so the model knows the expected format.

In [ ]:
from transformers import AutoTokenizer

print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"  # required by SFTTrainer

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token!r}")

In [ ]:
def format_medqa_sample(sample: dict, tokenizer) -> str:
    """
    MedQA sample format:
        input:  "A 45-year-old man presents with...\nA) Option A\nB) Option B\nC) Option C\nD) Option D"
        output: "The correct answer is B) ...\n\nExplanation: ..."

    We wrap this in a Mistral chat template with a medical system prompt.
    The system prompt tells the model to behave as a medical AI and always
    pick a specific answer letter — important for MCQ-style training.
    """
    system_msg = (
        "You are a knowledgeable medical AI assistant. "
        "When given a clinical multiple-choice question, analyze the case carefully, "
        "identify the correct answer (A, B, C, or D), and provide a clear explanation. "
        "Always begin your response with 'The correct answer is X)' where X is the letter."
    )
    
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": sample["input"]},
        {"role": "assistant", "content": sample["output"]},
    ]
    
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return formatted


# Test the formatter on one sample
sample_formatted = format_medqa_sample(raw_dataset[0], tokenizer)
print("Example formatted sample (first 600 chars):")
print(sample_formatted[:600])
print("...")

# Token count for that sample
token_count = len(tokenizer.encode(sample_formatted))
print(f"\nToken count: {token_count} (max_seq_length={MAX_SEQ_LENGTH})")

In [ ]:
from sklearn.model_selection import train_test_split

# ─── Step 1: Filter 5-option samples ─────────────────────────
dataset_filtered = raw_dataset.filter(lambda s: not has_5_options(s))
print(f"After filtering 5-option samples: {len(dataset_filtered)} samples")

# ─── Step 2: Train/test split ─────────────────────────────────
# We split BEFORE formatting so the test set can be used for MCQ evaluation
# (evaluation needs the raw input/output fields, not the formatted text)
split = dataset_filtered.train_test_split(test_size=TEST_SIZE, seed=42)
train_raw = split["train"]
test_raw  = split["test"]

print(f"Train samples: {len(train_raw)}")
print(f"Test samples:  {len(test_raw)}")

# ─── Step 3: Optional subsample for quick test runs ──────────
if MAX_SAMPLES is not None:
    train_raw = train_raw.select(range(min(MAX_SAMPLES, len(train_raw))))
    print(f"Subsampled to {len(train_raw)} training samples")

# ─── Step 4: Format training data ────────────────────────────
train_dataset = train_raw.map(
    lambda s: {"text": format_medqa_sample(s, tokenizer)},
    remove_columns=train_raw.column_names,
)

print(f"\nFormatted training dataset: {len(train_dataset)} samples")
print(f"Columns: {train_dataset.column_names}")

## 5. Load Model with 4-bit Quantization

QLoRA loads the 7B model in 4-bit NF4 format (~3.5GB) instead of float16 (~14GB).
This is what makes training on a single T4 possible.

## 5. Load Model with 4-bit Quantization

QLoRA loads the 7B model in 4-bit NF4 format (~3.5GB on disk) instead of float16 (~14GB).
This is what makes training on a single T4 (16GB VRAM) possible.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

# 4-bit NF4 quantization config — the heart of QLoRA.
# Without this, Mistral-7B needs ~14GB. With 4-bit: ~3.5GB → fits on T4.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {BASE_MODEL} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Casts layer norms to fp32 (stability), enables gradient checkpointing,
# unfreezes embeddings — required after 4-bit load before LoRA injection.
model = prepare_model_for_kbit_training(model)

print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 6. Inject LoRA Adapters

PEFT's `get_peft_model` freezes the base model weights and injects small trainable
LoRA matrices into the target modules. Only ~0.5% of parameters become trainable.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: trainable params ≈ 20M / 3750M (≈ 0.53%)

print(f"\nGPU memory after LoRA: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=False,
    bf16=True,                    # Mistral is bfloat16 — must match
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_32bit",    # memory-efficient optimizer for QLoRA
    gradient_checkpointing=True,  # trade compute for memory
    group_by_length=True,         # less padding waste
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

print(f"Training on {len(train_dataset)} samples for {NUM_EPOCHS} epochs")
steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM_STEPS)
print(f"Steps per epoch: ~{steps_per_epoch}")
print(f"Total steps:     ~{steps_per_epoch * NUM_EPOCHS}")
print("\nStarting training...")
trainer.train()

In [ ]:
# Save the LoRA adapter (just the adapter weights, not the full 7B model)
adapter_path = f"{OUTPUT_DIR}/final-adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"Adapter saved to: {adapter_path}")

# Check adapter size (should be ~50-200 MB, not 14 GB)
import os
total_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)
print(f"Adapter size: {total_size / 1e6:.1f} MB")

## 8. Plot Training Loss Curve

In [ ]:
import matplotlib.pyplot as plt

# Extract loss history from trainer
log_history = trainer.state.log_history
train_losses = [(e['step'], e['loss']) for e in log_history if 'loss' in e]

if train_losses:
    steps, losses = zip(*train_losses)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps, losses, color='steelblue', linewidth=1.5, alpha=0.8)
    ax.set_xlabel('Step')
    ax.set_ylabel('Training Loss')
    ax.set_title('Phase 2: Medical QLoRA — Training Loss')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/loss_curve.png", dpi=150)
    plt.show()
    
    print(f"Initial loss: {losses[0]:.4f}")
    print(f"Final loss:   {losses[-1]:.4f}")
    print(f"Reduction:    {(losses[0] - losses[-1]) / losses[0]:.1%}")
else:
    print("No loss history found.")

## 9. Qualitative Inference Check

Before running full MCQ evaluation, manually inspect a few responses to sanity-check quality.
Look for: correct answer letter format, coherent clinical reasoning, appropriate explanation depth.

In [ ]:
from peft import PeftModel

# Reload the adapter cleanly for inference
# (trainer.model has gradient checkpointing state — reload for clean eval)
print("Loading model + adapter for inference...")
eval_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
eval_model = PeftModel.from_pretrained(eval_model, adapter_path)
eval_model.eval()

print("Model ready for inference.")

In [ ]:
# Switch model to eval mode (disables dropout, fixes generation behavior)
model.eval()
eval_model = model

print("Model ready for inference.")

In [ ]:
import re
from tqdm.auto import tqdm

def extract_answer_letter(text: str) -> str:
    """
    Extract the predicted answer letter from the model's response.
    
    Handles formats like:
      - "The correct answer is A)"
      - "The correct answer is A."
      - "Answer: B"
      - "B) Pulmonary embolism" (when model jumps straight to the answer)
    
    Returns "?" if no letter found — these count as wrong answers.
    """
    text = text.strip()
    
    # Primary: "correct answer is X" pattern (what we trained for)
    m = re.search(r'correct answer is\s+([A-D])[).]?', text, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    
    # Fallback 1: "Answer: X" or "Answer is X"
    m = re.search(r'[Aa]nswer[:\s]+([A-D])[).]?', text)
    if m:
        return m.group(1).upper()
    
    # Fallback 2: first standalone letter at start of response
    m = re.match(r'^([A-D])[).\s]', text)
    if m:
        return m.group(1).upper()
    
    return "?"


def extract_ground_truth(output_text: str) -> str:
    """Extract ground truth letter from the MedQA output field."""
    m = re.search(r'correct answer is\s+([A-D])[).]?', output_text, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    # Some samples start with just the letter
    m = re.match(r'^([A-D])[).\s]', output_text.strip())
    if m:
        return m.group(1).upper()
    return "?"


# Verify extractor on a few ground truth samples
print("Ground truth extraction check:")
for i in range(5):
    gt = extract_ground_truth(test_raw[i]['output'])
    print(f"  Sample {i}: GT='{gt}' | Full output start: {test_raw[i]['output'][:60]}")

In [ ]:
# Full MCQ accuracy evaluation
# Runtime: ~MAX_EVAL_SAMPLES * 5 seconds ≈ 40 min for 500 samples on T4
# Reduce MAX_EVAL_SAMPLES (set to 100) for a quick check

eval_samples = test_raw.select(range(min(MAX_EVAL_SAMPLES, len(test_raw))))

correct = 0
total   = 0
no_answer = 0
skipped_gt = 0
results = []

for sample in tqdm(eval_samples, desc="Evaluating MCQ accuracy"):
    gt_letter   = extract_ground_truth(sample['output'])
    if gt_letter == "?":
        skipped_gt += 1
        continue  # skip samples where we can't parse ground truth
    
    response    = generate_medical(eval_model, tokenizer, sample['input'], max_new_tokens=200)
    pred_letter = extract_answer_letter(response)
    
    is_correct = (pred_letter == gt_letter)
    if pred_letter == "?":
        no_answer += 1
    
    correct += is_correct
    total   += 1
    results.append({
        "ground_truth": gt_letter,
        "predicted":    pred_letter,
        "correct":      is_correct,
    })

accuracy = correct / total if total > 0 else 0.0

print(f"\n{'='*50}")
print(f"MCQ EVALUATION RESULTS")
print(f"{'='*50}")
print(f"Samples evaluated:    {total}")
print(f"Skipped (no GT):      {skipped_gt}")
print(f"Correct:              {correct}")
if total > 0:
    print(f"Accuracy:             {accuracy:.1%}")
    print(f"No-answer rate:       {no_answer/total:.1%} ({no_answer} samples)")
else:
    print("⚠️  ZERO samples evaluated — ground truth extractor failed on all of them.")
    print("    Run the diagnostic cell to see what test_raw[i]['output'] looks like.")
print(f"{'='*50}")
print(f"Random chance baseline: 25.0%")
print(f"Target (base + 5pp):    ~50-55%")

In [ ]:
MEDICAL_SYSTEM = (
    "You are a knowledgeable medical AI assistant. "
    "When given a clinical multiple-choice question, analyze the case carefully, "
    "identify the correct answer (A, B, C, or D), and provide a clear explanation. "
    "Always begin your response with 'The correct answer is X)' where X is the letter."
)

def generate_medical(model, tokenizer, question: str, max_new_tokens: int = 200) -> str:
    messages = [
        {"role": "system", "content": MEDICAL_SYSTEM},
        {"role": "user",   "content": question},
    ]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][input_len:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# Quick sanity check
test_q = test_raw[0]['input']
print("Test question (first 200 chars):")
print(test_q[:200])
print("\nGenerating response...")
resp = generate_medical(eval_model, tokenizer, test_q)
print(f"\nResponse:\n{resp[:300]}")

In [ ]:
# Per-answer-letter breakdown — useful for spotting answer bias
import pandas as pd

df = pd.DataFrame(results)
print("Prediction distribution:")
print(df['predicted'].value_counts().to_string())
print()
print("Ground truth distribution:")
print(df['ground_truth'].value_counts().to_string())
print()

# Accuracy per ground truth letter
print("Accuracy by answer letter:")
for letter in ['A', 'B', 'C', 'D']:
    letter_df = df[df['ground_truth'] == letter]
    if len(letter_df) > 0:
        letter_acc = letter_df['correct'].mean()
        print(f"  {letter}: {letter_acc:.1%} ({len(letter_df)} samples)")

In [ ]:
# Save evaluation results to disk
import json

eval_summary = {
    "phase": "phase2_medical_medqa",
    "model": BASE_MODEL,
    "adapter": adapter_path,
    "dataset": DATASET_NAME,
    "num_train_samples": len(train_dataset),
    "num_eval_samples": total,
    "num_epochs": NUM_EPOCHS,
    "accuracy": round(accuracy, 4),
    "correct": correct,
    "no_answer_rate": round(no_answer / total, 4) if total > 0 else 0,
}

with open(f"{OUTPUT_DIR}/eval_results.json", "w") as f:
    json.dump(eval_summary, f, indent=2)

print("Evaluation summary saved to eval_results.json")
print(json.dumps(eval_summary, indent=2))

## 11. Push Adapter to HuggingFace Hub

Uploads just the adapter weights (~50-200MB), not the full 7B base model.

In [ ]:
# Get HF token from Kaggle secrets
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets.")
except Exception:
    # Fallback: paste token directly (don't commit this)
    hf_token = input("Enter your HuggingFace token: ").strip()

from huggingface_hub import login
login(token=hf_token)

In [ ]:
from huggingface_hub import HfApi

# Set your HF username here
HF_USERNAME = "anksriv"
REPO_NAME   = f"{HF_USERNAME}/mistral-7b-medical-medqa-qlora"

api = HfApi()
api.create_repo(repo_id=REPO_NAME, exist_ok=True, private=False)

# Push adapter weights (just LoRA, ~50-200 MB — not the full 7B model)
trainer.model.push_to_hub(REPO_NAME, token=hf_token)
tokenizer.push_to_hub(REPO_NAME, token=hf_token)

# Push eval results alongside
api.upload_file(
    path_or_fileobj=f"{OUTPUT_DIR}/eval_results.json",
    path_in_repo="eval_results.json",
    repo_id=REPO_NAME,
    token=hf_token,
)

# Push model card — honest README with partial-training caveat
# Reads from /kaggle/working/ if you copied it there, else from the repo path.
import os
model_card_paths = [
    "/kaggle/working/medical_README.md",
    "/kaggle/input/jobs-prjcts/03-llm-finetuning/model_cards/medical_README.md",
    "../model_cards/medical_README.md",
]
card_path = next((p for p in model_card_paths if os.path.exists(p)), None)

if card_path:
    api.upload_file(
        path_or_fileobj=card_path,
        path_in_repo="README.md",
        repo_id=REPO_NAME,
        token=hf_token,
    )
    print(f"Model card uploaded from {card_path}")
else:
    print("⚠️  model_cards/medical_README.md not found — upload it manually.")
    print("   Paths checked:", model_card_paths)

print(f"\nAdapter pushed to: https://huggingface.co/{REPO_NAME}")

## 12. Summary

| Item | Value |
|------|-------|
| Base model | `mistralai/Mistral-7B-Instruct-v0.2` |
| Dataset | `medalpaca/medical_meadow_medqa` |
| Training samples | ~9,000 |
| Epochs | 2 |
| LoRA rank | 16 |
| Max seq length | 1,024 |
| Eval metric | MCQ accuracy (A/B/C/D) |
| Target | ≥50% (base Mistral ~45%) |

**Next step:** Run `notebooks/03_legal.ipynb` for Phase 3 (legal contract analysis).